In [5]:
# Install required libraries

!pip install sentence-transformers
!pip install PyPDF2
!pip install python-docx
!pip install scikit-learn
!pip install joblib

In [6]:
import joblib
import numpy as np
import pandas as pd
import PyPDF2
from docx import Document

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from google.colab import files

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
MODEL_PATH = "/content/drive/MyDrive/resume_project/ats_score_model.pkl"

model = joblib.load(MODEL_PATH)

print("Model Loaded Successfully")

Model Loaded Successfully


In [25]:


embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [26]:


uploaded = files.upload()

Saving Aman_Resume.docx to Aman_Resume (1).docx


In [11]:
import os


def extract_resume_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        text = ""
        with open(file_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text

    elif ext == ".docx":
        doc = Document(file_path)
        return "\n".join([p.text for p in doc.paragraphs])

    else:
        raise ValueError("Only PDF and DOCX files are supported.")



In [12]:
resume_file = list(uploaded.keys())[0]

resume_text = extract_resume_text(resume_file)

print(resume_text[:10000])   # Preview first 1000 characters


OBJECTIVE
To obtain a challenging entry‑level position in a reputed organization where I can apply my skills in Python, machine learning, and data analysis to solve practical problems. I aim to learn emerging technologies, contribute to innovative projects, and grow alongside the company while continuously enhancing my knowledge in computer science.
EDUCATION
Hindustan College of Science and Technology, Mathura (2023-2027)                                                   
 B.Tech - Computer Science and Engineering - CGPA – 7.77                                                 
PROJECT
Medicine Recommendation System 
• Developed a machine learning model to recommend medicines based on patient symptoms, enhancing decision support. 
•  Implemented data preprocessing, feature engineering, and supervised classification techniques to improve model accuracy.
•  Optimized training datasets, achieving measurable gains in prediction performance.
• Tools used: Python, Scikit-learn, Pandas. 

INT

In [13]:
job_description ='''Paste your job description here.
For example:

Looking for a Python Developer with experience in
Machine Learning, SQL, Flask, Pandas, NumPy,
REST APIs, Git, and Data Structures.
'''

In [14]:
resume_embedding = embedding_model.encode(resume_text)

job_embedding = embedding_model.encode(job_description)

print("Resume Embedding Shape :", resume_embedding.shape)
print("Job Embedding Shape    :", job_embedding.shape)

Resume Embedding Shape : (384,)
Job Embedding Shape    : (384,)


In [15]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_score = cosine_similarity(
    [resume_embedding],
    [job_embedding]
)[0][0]

print("Cosine Similarity :", round(cosine_score, 4))

Cosine Similarity : 0.6484


In [16]:
skills_df = pd.read_csv("/content/drive/MyDrive/resume_project/skills_taxonomy.csv")
print(skills_df.columns.tolist())   # check the actual column name first, don't assume
print(skills_df.head())

['skill', 'category']
        skill               category
0      python  programming_languages
1        java  programming_languages
2  javascript  programming_languages
3  typescript  programming_languages
4         c++  programming_languages


In [17]:
!pip install spacy
import spacy
from spacy.matcher import PhraseMatcher

SKILL_TAXONOMY = set(skills_df["skill"].str.lower().str.strip())  # adjust column name if needed

nlp = spacy.blank("en")
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
patterns = [nlp.make_doc(skill) for skill in SKILL_TAXONOMY]
matcher.add("SKILLS", patterns)

def extract_skills(text):
    doc = nlp(text)
    return {doc[start:end].text.lower() for _, start, end in matcher(doc)}

resume_skills = extract_skills(resume_text)
jd_skills = extract_skills(job_description)

skill_overlap_v2 = len(resume_skills & jd_skills) / len(jd_skills) if jd_skills else 0.0
matched_skills = sorted(resume_skills & jd_skills)
missing_skills = sorted(jd_skills - resume_skills)

print("Matched skills:", matched_skills)
print("Missing skills:", missing_skills)
print("Skill overlap:", round(skill_overlap_v2, 3))

Matched skills: ['machine learning', 'numpy', 'pandas', 'python']
Missing skills: ['flask', 'git', 'sql']
Skill overlap: 0.571


In [18]:
import re

def parse_min_experience(text):
    nums = [int(n) for n in re.findall(r'\d+', text)]
    return float(min(nums)) if nums else 0.0

experience_required_min = parse_min_experience(job_description)
print("Experience required (min years):", experience_required_min)

Experience required (min years): 0.0


In [19]:
# Counts date-range patterns like "2019 - 2021" or "2020 - Present" as a rough proxy for number of roles
date_pattern = r'(19|20)\d{2}\s*[-–—]\s*((19|20)\d{2}|Present|present|Current|current)'
num_positions_held = len(re.findall(date_pattern, resume_text))
if num_positions_held == 0:
    num_positions_held = 1   # assume at least one role if no dates were detected

print("Estimated positions held:", num_positions_held)

Estimated positions held: 1


In [20]:
LEVELS = {"phd":5,"doctorate":5,"master":4,"mba":4,"bachelor":3,"associate":2,"diploma":2,"certificate":1,"high school":0}

def extract_level(text):
    text = text.lower()
    found = [v for k, v in LEVELS.items() if k in text]
    return max(found) if found else None

candidate_level = extract_level(resume_text)
required_level = extract_level(job_description)

education_level_match = int(
    candidate_level is not None and required_level is not None and candidate_level >= required_level
)
print("Education level match:", education_level_match)

Education level match: 0


In [21]:
# field_match was your weakest feature (lowest correlation in training) - a simple default is
# a reasonable trade-off here rather than building fragile text-matching logic for one weak signal
field_match = 0

In [22]:
feature_names = joblib.load("/content/drive/MyDrive/resume_project/feature_names.pkl")

feature_values = {
    "cosine_similarity": cosine_score,
    "skill_overlap_v2": skill_overlap_v2,
    "experience_required_min": experience_required_min,
    "num_positions_held": num_positions_held,
    "education_level_match": education_level_match,
    "field_match": field_match,
}

X_new = pd.DataFrame([[feature_values[f] for f in feature_names]], columns=feature_names)
print(X_new)

predicted_ats_score = model.predict(X_new)[0] * 100
print("Predicted ATS Score:", round(predicted_ats_score, 2))

   cosine_similarity  skill_overlap_v2  experience_required_min  \
0           0.648362          0.571429                      0.0   

   num_positions_held  education_level_match  field_match  
0                   1                      0            0  
Predicted ATS Score: 83.05


In [23]:
def get_embedding_safe(text, model, chunk_size=200):
    words = text.split()
    if len(words) <= chunk_size:
        return model.encode(text)
    # Split into chunks, embed each, then average - captures the WHOLE document
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    chunk_embeddings = model.encode(chunks)
    return np.mean(chunk_embeddings, axis=0)

resume_embedding = get_embedding_safe(resume_text, embedding_model)
job_embedding = get_embedding_safe(job_description, embedding_model)

In [24]:
resume_text = extract_resume_text(resume_file)

if not resume_text.strip():
    raise ValueError("No text extracted - this PDF might be a scanned image (needs OCR), not real text.")

print(f"Extracted {len(resume_text.split())} words.")

Extracted 188 words.


In [27]:
existing_candidates = pd.read_csv("/content/drive/MyDrive/resume_project/resume_jd_ranked_final.csv")
print(existing_candidates["job_position_name"].unique())

['AI Engineer' 'Asst. Manager/ Manger (Administrative)'
 'Business Development Executive' 'Civil Engineer' 'Data Engineer'
 'Data Science Engineer' 'Database Administrator (DBA)' 'DevOps Engineer'
 'Executive - VAT'
 'Executive/ Senior Executive- Trade Marketing, Hygiene Products'
 'HR Officer' 'Head of Internal Control & Compliance (ICC) - SEVP/DMD'
 'Management Trainee - Mechanical'
 'Manager- Human Resource Management (HRM)\n' 'Marketing Officer'
 'Mechanical Designer' 'Mechanical Engineer' 'Network Support Engineer'
 'Project Coordinator (Civil)' 'Senior iOS Engineer' 'Site Engineer'
 'Sr.Officer / Executive - Internal Audit'
 'System Administrator (Operation & Maintenance of Server, Storage & Service Desk System)']


In [28]:
target_job = "AI Engineer"   # <-- paste the exact spelling from the list printed above

In [29]:
job_pool = existing_candidates[existing_candidates["job_position_name"] == target_job]
existing_scores = sorted(job_pool["ats_score"].tolist(), reverse=True)

# Count how many existing candidates scored higher than this new resume
rank = sum(1 for s in existing_scores if s > predicted_ats_score) + 1
total_candidates = len(existing_scores) + 1   # +1 to include this new resume

print(f"This resume would rank #{rank} out of {total_candidates} for '{target_job}'")

This resume would rank #8 out of 171 for 'AI Engineer'


In [30]:
print("="*50)
print("RESUME SCREENING RESULT")
print("="*50)
print(f"Target Job     : {target_job}")
print(f"ATS Score       : {round(predicted_ats_score, 2)} / 100")
print(f"Resume Rank     : #{rank} out of {total_candidates}")
print(f"Matched Skills  : {matched_skills}")
print(f"Missing Skills  : {missing_skills}")
print("="*50)

RESUME SCREENING RESULT
Target Job     : AI Engineer
ATS Score       : 83.05 / 100
Resume Rank     : #8 out of 171
Matched Skills  : ['machine learning', 'numpy', 'pandas', 'python']
Missing Skills  : ['flask', 'git', 'sql']
